# Notebook 05 - Coastal Water Quality Feature Mapping

## Objective

This notebook uses Planet Tanager hyperspectral surface reflectance to investigate spatial variations in the optical properties of coastal waters in Corozal Bay, Belize.

The analysis focuses on relative spectral indicators rather than absolute concentrations of water-quality parameters because no coincident field measurements were taken for calibration.

## Planned analysis

1. Load the Tanager hyperspectral cube and metadata.
2. Apply the previously developed coastal water mask.
3. Remove unreliable spectral wavelengths.
4. Calculate relative spectral indicators.
5. Map their spatial distribution
6. Compare the indicators with RGB imagery and spectral classes.

## Importanst limitation

The resulting maps represent relative spectral patterns and should not be interpreted as calibrated measurements of turbidity, chlorophyll-a, or suspended sediment concentration without independent field observations or a validated retrieval model.

In [2]:
#01_Import Data
import h5py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams["figure.figsize"] = (10, 7)
plt.rcParams["axes.grid"] = False

DATA_PATH = Path(r"C:\GIS_Work\5 GeoSpectra AI\data\raw\20250824_171857_84_4001_ortho_sr_hdf5.h5")

OUTPUT_DIR = Path(r"C:\GIS_Work\5 GeoSpectra AI\outputs\figures")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Environment ready.")

Environment ready.


In [3]:
#02_Load the Tanager data
with h5py.File(DATA_PATH, "r") as f:
    
    sr_dataset = f["HDFEOS/GRIDS/HYP/Data Fields/surface_reflectance"]

    cube = sr_dataset[:]

    wavelengths = sr_dataset.attrs["wavelengths"]

    good_wavelengths = sr_dataset.attrs["good_wavelengths"]

    cloud_mask = f["HDFEOS/GRIDS/HYP/Data Fields/beta_cloud_mask"][:]

    cirrus_mask = f["HDFEOS/GRIDS/HYP/Data Fields/beta_cirrus_mask"][:]

    nodata_mask = f["HDFEOS/GRIDS/HYP/Data Fields/nodata_pixels"][:]
    
cube = cube.astype(np.float32)

cube[cube == -9999.0] = np.nan

good_wavelengths = good_wavelengths.astype(bool)

print("Cube shape:", cube.shape)
print("Number of wavelengths:", len(wavelengths))
print("Good wavelengths:", np.sum(good_wavelengths))

Cube shape: (426, 671, 763)
Number of wavelengths: 426
Good wavelengths: 368


In [11]:
#03_Understand the water mask
WATER_MASK_PATH = Path(r"C:\GIS_Work\5 GeoSpectra AI\outputs\maps\water_mask.npy")

water_mask = np.load(WATER_MASK_PATH)
print("Water mask shape:", water_mask.shape)
print("Water pixels:", np.sum(water_mask))
print("Water size:", water_mask.size)
print("Water percentage:", np.sum(water_mask)/water_mask.size * 100)

Water mask shape: (671, 763)
Water pixels: 103239
Water size: 511973
Water percentage: 20.16493057251066


In [31]:
#04_Create a clean analysis mask
valid_water = (water_mask
               & (cloud_mask == 0)
               & (cirrus_mask == 0)
               & (nodata_mask == 0)
              )

print("Valid water pixels:", np.sum(valid_water))

Valid water pixels: 103239


In [32]:
#05_Find spectral bands

def find_band(target_nm):
    return np.argmin(np.abs(wavelengths - target_nm))

blue_490 = find_band(490)
green_560 = find_band(560)
red_665 = find_band(665)
red_edge_705 = find_band(705)
nir_865 = find_band(865)

print("Selected spectral bands")
print("="*60)

print(f"Blue     : Band {blue_490} | {wavelengths[blue_490]:.2f} nm")
print(f"Green    : Band {green_560} | {wavelengths[green_560]:.2f} nm")
print(f"Red      : Band {red_665} | {wavelengths[red_665]:.2f} nm")
print(f"Red Edge : Band {red_edge_705} | {wavelengths[red_edge_705]:.2f} nm")
print(f"NIR      : Band {nir_865} | {wavelengths[nir_865]:.2f} nm")

Selected spectral bands
Blue     : Band 23 | 490.93 nm
Green    : Band 37 | 560.83 nm
Red      : Band 58 | 665.87 nm
Red Edge : Band 66 | 705.92 nm
NIR      : Band 98 | 866.28 nm


In [35]:
#06_Check wavelength quality

selected_bands = {"Blue": blue_490,
                  "Green": green_560,
                  "Red": red_665,
                  "Red Edge": red_edge_705,
                  "NIR": nir_865}

print("Wavelength Quality Check")
print("=" * 60)

for name, band in selected_bands.items():

    status = "GOOD" if good_wavelengths[band] else "FLAGGED"

    print(f"{name:10s} | "f"Band {band:3d} | "f"{wavelengths[band]:8.2f} nm |"
         f"{status}")

Wavelength Quality Check
Blue       | Band  23 |   490.93 nm |GOOD
Green      | Band  37 |   560.83 nm |GOOD
Red        | Band  58 |   665.87 nm |GOOD
Red Edge   | Band  66 |   705.92 nm |GOOD
NIR        | Band  98 |   866.28 nm |GOOD


In [42]:
#07_Extract spectral layers

blue_layer = cube[blue_490]
green_layer = cube[green_560]
red_layer = cube[red_665]
red_edge_layer = cube[red_edge_705]
nir_layer = cube[nir_865]

print("Spectral layers extracted.")
print("Layer shape:", nir_layer.shape)


Spectral layers extracted.
Layer shape: (671, 763)


In [45]:
#08_Apply water mask

blue_water = np.where(valid_water, blue_layer, np.nan)
green_water = np.where(valid_water, green_layer, np.nan)
red_water = np.where(valid_water, red_layer, np.nan)
red_edge_water = np.where(valid_water, red_edge_layer, np.nan)
nir_water = np.where(valid_water, nir_layer, np.nan)

print("Water-only spectral layers created.")


Water-only spectral layers created.


In [55]:
#09_Calculate relative spectral indicators

green_blue_ratio = green_water / blue_water

red_green_ratio = red_water / green_water

nir_red_ratio = nir_water / red_water

print("Spectral indicators calculated.")

Spectral indicators calculated.


In [76]:
#10_Clean spectral indicators

def clean_ratio(layer, lower=1, upper=99):

    layer = layer.astype(np.float32).copy()

    layer[~np.isfinite(layer)] = np.nan

    low = np.nanpercentile(layer, lower)
    high = np.nanpercentile(layer, upper)

    layer[
        (layer < low) |
        (layer > high)
    ] = np.nan

    return layer


green_blue_ratio_clean = clean_ratio(
    green_blue_ratio
)

red_green_ratio_clean = clean_ratio(
    red_green_ratio
)

nir_red_ratio_clean = clean_ratio(
    nir_red_ratio
)

print("Spectral indicators cleaned.")

Spectral indicators cleaned.


In [78]:
#11_Verify spectral indicator cleaning

def verify_cleaning(name, original, cleaned):

    original_values = original[np.isfinite(original)]
    cleaned_values = cleaned[np.isfinite(cleaned)]

    original_low = np.percentile(original_values, 1)
    original_high = np.percentile(original_values, 99)

    print(f"\n{name}")
    print("=" * 60)

    print(f"Original valid pixels : {len(original_values):,}")
    print(f"Cleaned valid pixels  : {len(cleaned_values):,}")

    print(f"\nOriginal 1% threshold : {original_low:.4f}")
    print(f"Original 99% threshold: {original_high:.4f}")

    print(f"\nCleaned minimum       : {np.min(cleaned_values):.4f}")
    print(f"Cleaned median        : {np.median(cleaned_values):.4f}")
    print(f"Cleaned 99%           : {np.percentile(cleaned_values, 99):.4f}")
    print(f"Cleaned maximum       : {np.max(cleaned_values):.4f}")

    print(
        f"\nValues above original 99% threshold: "
        f"{np.sum(cleaned_values > original_high):,}"
    )


verify_cleaning(
    "Green / Blue Ratio",
    green_blue_ratio,
    green_blue_ratio_clean
)

verify_cleaning(
    "Red / Green Ratio",
    red_green_ratio,
    red_green_ratio_clean
)

verify_cleaning(
    "NIR / Red Ratio",
    nir_red_ratio,
    nir_red_ratio_clean
)


Green / Blue Ratio
Original valid pixels : 103,239
Cleaned valid pixels  : 101,173

Original 1% threshold : 1.0408
Original 99% threshold: 1.4280

Cleaned minimum       : 1.0408
Cleaned median        : 1.2213
Cleaned 99%           : 1.3730
Cleaned maximum       : 1.4278

Values above original 99% threshold: 0

Red / Green Ratio
Original valid pixels : 103,239
Cleaned valid pixels  : 101,173

Original 1% threshold : 0.5888
Original 99% threshold: 0.8987

Cleaned minimum       : 0.5888
Cleaned median        : 0.7331
Cleaned 99%           : 0.8874
Cleaned maximum       : 0.8987

Values above original 99% threshold: 0

NIR / Red Ratio
Original valid pixels : 103,239
Cleaned valid pixels  : 101,173

Original 1% threshold : 0.6717
Original 99% threshold: 4.3786

Cleaned minimum       : 0.6717
Cleaned median        : 0.9210
Cleaned 99%           : 3.1615
Cleaned maximum       : 4.3777

Values above original 99% threshold: 0
